# Nifty 750 — OHLCV Downloader (Option A)

Downloads daily OHLCV data for the **current Nifty Total Market / 750-stock constituent list** from **1 January 2021 through the present date** using Yahoo Finance via `yfinance`.

> **Option A:** the same current 750-stock universe is used for the entire historical period. This is not a historical-constituent/survivorship-bias-free dataset.

### Input
`nifty_total_market_constituents.csv`

The CSV should contain a column named `Symbol` (or one of the common alternatives handled below).

### Outputs
- `nifty_750_OHLCV_2021_present.csv`
- `failed_tickers.csv`


In [1]:
# Install required packages
!pip install -q yfinance pandas tqdm

In [4]:
import os
import time
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

START_DATE = "2021-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

INPUT_FILE = "ind_niftytotalmarket_list.csv"
OUTPUT_FILE = "nifty_750_OHLCV_2021_present.csv"
FAILED_FILE = "failed_tickers.csv"

print("Start date:", START_DATE)
print("End date  :", END_DATE)


Start date: 2021-01-01
End date  : 2026-09-29


## 1. Load the current Nifty 750 constituent file

Place `nifty_total_market_constituents.csv` in the **same folder as this notebook**.

The code automatically looks for common symbol-column names such as `Symbol`, `SYMBOL`, `Ticker`, etc.


In [5]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Could not find '{INPUT_FILE}'. "
        "Place the CSV in the same folder as this notebook."
    )

constituents = pd.read_csv(INPUT_FILE)

print("Columns found:")
print(constituents.columns.tolist())
print("\nNumber of rows:", len(constituents))


Columns found:
['Company Name', 'Industry', 'Symbol', 'Series', 'ISIN Code']

Number of rows: 755


In [6]:
possible_columns = [
    "Symbol", "SYMBOL", "symbol",
    "Ticker", "ticker",
    "NSE Symbol", "NSE_SYMBOL",
    "NSE_Symbol"
]

symbol_column = next(
    (col for col in possible_columns if col in constituents.columns),
    None
)

if symbol_column is None:
    raise ValueError(
        "No NSE symbol column was found. "
        "Rename your symbol column to 'Symbol' and rerun this cell."
    )

symbols = (
    constituents[symbol_column]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"NAN": np.nan})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Symbol column:", symbol_column)
print("Unique symbols:", len(symbols))
print("\nFirst 20 symbols:")
print(symbols[:20])


Symbol column: Symbol
Unique symbols: 755

First 20 symbols:
['360ONE', '3MINDIA', 'ABB', 'ACC', 'ACMESOLAR', 'AIAENG', 'APLAPOLLO', 'ASKAUTOLTD', 'AUBANK', 'AWL', 'AXISCADES', 'AADHARHFC', 'AARTIDRUGS', 'AARTIIND', 'AARTIPHARM', 'AAVAS', 'ABBOTINDIA', 'ACE', 'ACUTAAS', 'ADANIENSOL']


## 2. Convert NSE symbols to Yahoo Finance tickers

NSE symbols are converted as follows:

`RELIANCE` → `RELIANCE.NS`

`TCS` → `TCS.NS`


In [7]:
yahoo_tickers = [f"{symbol}.NS" for symbol in symbols]

ticker_map = dict(zip(yahoo_tickers, symbols))

print("First 20 Yahoo Finance tickers:")
print(yahoo_tickers[:20])


First 20 Yahoo Finance tickers:
['360ONE.NS', '3MINDIA.NS', 'ABB.NS', 'ACC.NS', 'ACMESOLAR.NS', 'AIAENG.NS', 'APLAPOLLO.NS', 'ASKAUTOLTD.NS', 'AUBANK.NS', 'AWL.NS', 'AXISCADES.NS', 'AADHARHFC.NS', 'AARTIDRUGS.NS', 'AARTIIND.NS', 'AARTIPHARM.NS', 'AAVAS.NS', 'ABBOTINDIA.NS', 'ACE.NS', 'ACUTAAS.NS', 'ADANIENSOL.NS']


## 3. Download OHLCV data

The downloader uses `auto_adjust=False` so that the raw `Open`, `High`, `Low`, and `Close` fields are retained, alongside `Adj Close`.

The download is performed in batches to make the process more robust for a large universe.


In [8]:
BATCH_SIZE = 50
SLEEP_SECONDS = 2

all_batches = []
downloaded_tickers = []
failed_tickers = []

for start_idx in tqdm(
    range(0, len(yahoo_tickers), BATCH_SIZE),
    desc="Downloading ticker batches"
):
    batch = yahoo_tickers[start_idx:start_idx + BATCH_SIZE]

    try:
        batch_data = yf.download(
            tickers=batch,
            start=START_DATE,
            end=END_DATE,
            interval="1d",
            auto_adjust=False,
            repair=True,
            group_by="ticker",
            threads=True,
            progress=False
        )

        if batch_data is None or batch_data.empty:
            failed_tickers.extend(batch)
            continue

        all_batches.append(batch_data)

        # Identify tickers that actually returned data
        if isinstance(batch_data.columns, pd.MultiIndex):
            returned = set(batch_data.columns.get_level_values(0))
            for ticker in batch:
                if ticker in returned:
                    downloaded_tickers.append(ticker)
                else:
                    failed_tickers.append(ticker)
        else:
            # This occurs when a single ticker is returned
            downloaded_tickers.append(batch[0])

    except Exception as e:
        print(f"Batch starting at {start_idx} failed: {e}")
        failed_tickers.extend(batch)

    time.sleep(SLEEP_SECONDS)

print("\nBatches completed:", len(all_batches))
print("Tickers with returned data:", len(set(downloaded_tickers)))
print("Tickers initially flagged as failed:", len(set(failed_tickers)))


C:\Users\maina\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
C:\Users\maina\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
C:\Users\maina\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of


Batches completed: 16
Tickers with returned data: 755
Tickers initially flagged as failed: 0


## 4. Convert downloaded data to long format

The final dataset has one row per **stock-date**:

`Date | Symbol | Yahoo_Ticker | Open | High | Low | Close | Adj_Close | Volume`

This format is convenient for A/D calculations and market-regime features.


In [9]:
if not all_batches:
    raise ValueError(
        "No data was downloaded. Check your internet connection, "
        "CSV symbols, and Yahoo Finance availability."
    )

records = []

for batch_data in all_batches:
    if isinstance(batch_data.columns, pd.MultiIndex):
        batch_tickers = batch_data.columns.get_level_values(0).unique()

        for ticker in batch_tickers:
            try:
                stock_data = batch_data[ticker].copy()
                stock_data = stock_data.reset_index()

                stock_data["Yahoo_Ticker"] = ticker
                stock_data["Symbol"] = ticker.replace(".NS", "")

                records.append(stock_data)

            except Exception as e:
                print(f"Could not process {ticker}: {e}")

    else:
        # Defensive handling for a one-ticker DataFrame
        stock_data = batch_data.copy().reset_index()
        ticker = None

        if len(downloaded_tickers) == 1:
            ticker = downloaded_tickers[0]

        if ticker is not None:
            stock_data["Yahoo_Ticker"] = ticker
            stock_data["Symbol"] = ticker.replace(".NS", "")
            records.append(stock_data)

if not records:
    raise ValueError("Downloaded data could not be converted into records.")

final_data = pd.concat(records, ignore_index=True)

# Normalize column names
final_data.columns = [
    str(col).replace(" ", "_")
    for col in final_data.columns
]

if "Adj_Close" not in final_data.columns and "Adj Close" in final_data.columns:
    final_data.rename(columns={"Adj Close": "Adj_Close"}, inplace=True)

required_columns = [
    "Date", "Symbol", "Yahoo_Ticker",
    "Open", "High", "Low", "Close",
    "Adj_Close", "Volume"
]

missing = [c for c in required_columns if c not in final_data.columns]

if missing:
    raise ValueError(f"Missing expected columns: {missing}")

final_data = final_data[required_columns].copy()

# Remove rows where all OHLC fields are missing
final_data.dropna(
    subset=["Open", "High", "Low", "Close"],
    how="all",
    inplace=True
)

final_data.sort_values(
    ["Date", "Symbol"],
    inplace=True
)

final_data.reset_index(drop=True, inplace=True)

print("Final dataset shape:", final_data.shape)
display(final_data.head())


Final dataset shape: (926909, 9)


,Date,Symbol,Yahoo_Ticker,Open,High,Low,Close,Adj_Close,Volume
0,2021-01-01,360ONE,360ONE.NS,254.199997,258.750000,252.012497,256.162506,219.059433,37996.0
1,2021-01-01,3MINDIA,3MINDIA.NS,20830.699219,21243.949219,20830.699219,21127.599609,19347.394531,1663.0
2,2021-01-01,AARTIDRUGS,AARTIDRUGS.NS,710.000000,747.500000,708.500000,735.000000,724.708252,1450044.0
3,2021-01-01,AARTIIND,AARTIIND.NS,622.000000,637.500000,621.049988,630.250000,619.883972,812396.0
4,2021-01-01,AAVAS,AAVAS.NS,1703.000000,1736.449951,1685.150024,1720.650024,1720.650024,18196.0


## 5. Validate the downloaded dataset

This checks how many of the requested current constituents actually produced data.


In [10]:
requested_symbols = set(symbols)
downloaded_symbols = set(final_data["Symbol"].dropna().unique())

failed_symbols = sorted(requested_symbols - downloaded_symbols)

failed_df = pd.DataFrame({"Symbol": failed_symbols})

print("Requested unique stocks :", len(requested_symbols))
print("Downloaded unique stocks:", len(downloaded_symbols))
print("Failed/missing stocks   :", len(failed_symbols))
print("Total OHLCV records     :", len(final_data))

print("\nDate range:")
print("First:", final_data["Date"].min())
print("Last :", final_data["Date"].max())

print("\nSample stock counts:")
print(final_data.groupby("Symbol").size().sort_values().head())


Requested unique stocks : 755
Downloaded unique stocks: 750
Failed/missing stocks   : 5
Total OHLCV records     : 926909

Date range:
First: 2021-01-01 00:00:00
Last : 2026-09-28 00:00:00

Sample stock counts:
Symbol
ICICIAMC     195
PARKHOSPS    197
CORONA       198
WAKEFIT      198
SKFINDUS     202
dtype: int64


In [11]:
# Save the combined OHLCV file
final_data.to_csv(
    OUTPUT_FILE,
    index=False
)

# Save failed tickers for investigation/retry
failed_df.to_csv(
    FAILED_FILE,
    index=False
)

print("Saved:")
print("1.", OUTPUT_FILE)
print("2.", FAILED_FILE)


Saved:
1. nifty_750_OHLCV_2021_present.csv
2. failed_tickers.csv


## 6. Inspect a specific company

For example, to inspect Reliance:


In [12]:
reliance = final_data[
    final_data["Symbol"] == "RELIANCE"
].copy()

display(reliance.head())
display(reliance.tail())

print("Reliance trading rows:", len(reliance))


,Date,Symbol,Yahoo_Ticker,Open,High,Low,Close,Adj_Close,Volume
414,2021-01-01,RELIANCE,RELIANCE.NS,917.461670,921.615173,914.692688,917.230957,897.047058,10015175.0
962,2021-01-04,RELIANCE,RELIANCE.NS,920.738342,922.492004,908.231689,918.776978,898.559143,24513534.0
1510,2021-01-05,RELIANCE,RELIANCE.NS,908.693176,915.431091,902.693665,907.354858,887.388306,24123091.0
2058,2021-01-06,RELIANCE,RELIANCE.NS,907.262512,907.308655,879.226440,883.426086,863.986084,46401468.0
2606,2021-01-07,RELIANCE,RELIANCE.NS,886.310425,897.617188,879.226440,881.995422,862.586914,32325918.0


,Date,Symbol,Yahoo_Ticker,Open,High,Low,Close,Adj_Close,Volume
923728,2026-09-22,RELIANCE,RELIANCE.NS,1247.599976,1251.900024,1237.400024,1240.400024,1240.400024,10684376.0
924478,2026-09-23,RELIANCE,RELIANCE.NS,1242.000000,1252.800049,1238.900024,1248.000000,1248.000000,8352048.0
925228,2026-09-24,RELIANCE,RELIANCE.NS,1236.599976,1241.599976,1219.199951,1219.199951,1219.199951,13923795.0
925978,2026-09-25,RELIANCE,RELIANCE.NS,1210.500000,1227.400024,1210.500000,1226.000000,1226.000000,13138735.0
926728,2026-09-28,RELIANCE,RELIANCE.NS,1218.000000,1219.699951,1197.000000,1197.599976,1197.599976,13602349.0


Reliance trading rows: 1423


## 7. Optional: save one CSV per company

The combined CSV above is the main output. If you also want individual files for every company, run the following cell.

This creates a folder called `individual_stocks`.


In [ ]:
SAVE_INDIVIDUAL_FILES = False

if SAVE_INDIVIDUAL_FILES:
    output_dir = "individual_stocks"
    os.makedirs(output_dir, exist_ok=True)

    for symbol, stock_df in tqdm(
        final_data.groupby("Symbol"),
        desc="Saving individual stock files"
    ):
        safe_symbol = "".join(
            ch for ch in str(symbol)
            if ch.isalnum() or ch in ("_", "-")
        )

        path = os.path.join(
            output_dir,
            f"{safe_symbol}_OHLCV_2021_present.csv"
        )

        stock_df.to_csv(path, index=False)

    print(f"Individual files saved in: {output_dir}")
else:
    print("Skipped individual files. Set SAVE_INDIVIDUAL_FILES = True to enable.")


## Important note for Option A

This notebook intentionally uses the **current constituent list** for the complete 2021–present period.

Therefore, it is appropriate for your requested **Option A** dataset, but it should not be interpreted as the historical Nifty Total Market universe for each date.

For your later daily Advance/Decline calculation, this creates a **survivorship-bias issue** because companies that are in today's universe are treated as though they were part of the universe in 2021.
